Method: Dual-Layer Markov Transitions. This is the advanced version. It predicts the next location primarily based on the user's previous state (where they were just a moment ago, capturing sequential movement). If the trajectory breaks (no historical match for that specific movement), it uses a fallback layer: returning to the user's most frequent location for that time (which is essentially relying on the logic of Model 1 to fill in the gaps).

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

path = '/home/z5562390/work/dataset2023/dataset1.csv.gz'
output_path = 'Math(markov).csv'

# --- Dual-Layer Spatiotemporal Markov Model ---
def process_single_user_unified(user_df):
    uid = user_df['uid'].iloc[0]
    
    # 1. Prepare training data (Days 0-59) and pad to a complete spatiotemporal grid
    train_df = user_df[user_df['d'] <= 59].drop_duplicates(subset=['d', 't'], keep='last').copy()
    if train_df.empty: return pd.DataFrame()

    days, ts = range(0, 60), range(48)
    full_index = pd.MultiIndex.from_product([[uid], days, ts], names=['uid', 'd', 't'])
    train_full = train_df.set_index(['uid', 'd', 't']).reindex(full_index)
    train_full = train_full.ffill().bfill().reset_index()
    train_full['weekday'] = train_full['d'] % 7

    # 2. Build state transitions
    train_full['prev_x'] = train_full['x'].shift(1)
    train_full['prev_y'] = train_full['y'].shift(1)
    transitions = train_full.dropna(subset=['prev_x', 'prev_y']).copy()

    # Layer 1: Spatiotemporal Markov transitions (prev state -> next state)
    matrix_full = transitions.groupby(['weekday', 't', 'prev_x', 'prev_y', 'x', 'y']).size().reset_index(name='count')
    best_full = matrix_full.sort_values('count', ascending=False).drop_duplicates(subset=['weekday', 't', 'prev_x', 'prev_y'])

    # Layer 2: Fallback temporal patterns (most frequent location per time/weekday)
    matrix_temp = train_full.groupby(['weekday', 't', 'x', 'y']).size().reset_index(name='count')
    best_temp = matrix_temp.sort_values('count', ascending=False).drop_duplicates(subset=['weekday', 't'])

    # 3. Recursive prediction for Days 60-74
    curr_x, curr_y = train_full.iloc[-1]['x'], train_full.iloc[-1]['y']
    results = []

    for d in range(60, 75):
        weekday = d % 7
        for t in range(48):
            # Try Layer 1: Spatial transition matching
            match = best_full[(best_full['weekday'] == weekday) & (best_full['t'] == t) & 
                              (best_full['prev_x'] == curr_x) & (best_full['prev_y'] == curr_y)]
            
            if not match.empty:
                next_x, next_y = match.iloc[0]['x'], match.iloc[0]['y']
            else:
                # Fallback to Layer 2: Temporal habit (guaranteed match due to grid padding)
                match_t = best_temp[(best_temp['weekday'] == weekday) & (best_temp['t'] == t)]
                next_x, next_y = match_t.iloc[0]['x'], match_t.iloc[0]['y']
            
            results.append([uid, d, t, int(next_x), int(next_y)])
            curr_x, curr_y = next_x, next_y # Update state for next iteration

    return pd.DataFrame(results, columns=['uid', 'd', 't', 'x', 'y'])

# --- Streaming Execution Engine ---
# Initialize CSV with headers
pd.DataFrame(columns=['uid', 'd', 't', 'x', 'y']).to_csv(output_path, index=False)
dtypes = {'uid': np.int32, 'd': np.int16, 't': np.int8, 'x': np.int16, 'y': np.int16}
buffer_df = pd.DataFrame()

TARGET_USERS = 20000
user_count, stop_reading = 0, False

print(f"Starting   Dual-Layer Markov Model for {TARGET_USERS} users...")

# Process large CSV in chunks to manage memory
for chunk in pd.read_csv(path, usecols=['uid', 'd', 't', 'x', 'y'], dtype=dtypes, chunksize=500000):
    if stop_reading: break
    chunk = pd.concat([buffer_df, chunk], ignore_index=True)
    unique_uids = chunk['uid'].unique()
    
    # Buffer if chunk contains only one user to avoid splitting their data
    if len(unique_uids) == 1: 
        buffer_df = chunk
        continue
    
    # Buffer the last user's data to ensure completeness in the next chunk
    last_uid = unique_uids[-1]
    complete_data, buffer_df = chunk[chunk['uid'] != last_uid], chunk[chunk['uid'] == last_uid]
    
    for uid, user_data in complete_data.groupby('uid'):
        if user_count >= TARGET_USERS: 
            stop_reading = True
            break
        
        user_pred = process_single_user_unified(user_data)
        if not user_pred.empty:
            user_pred.to_csv(output_path, mode='a', header=False, index=False)
            
        user_count += 1
        if user_count % 500 == 0: print(f"Progress: {user_count} / {TARGET_USERS}...")

# Process remaining buffered data if target isn't met
if not stop_reading and not buffer_df.empty and user_count < TARGET_USERS:
    for uid, user_data in buffer_df.groupby('uid'):
        user_pred = process_single_user_unified(user_data)
        if not user_pred.empty: 
            user_pred.to_csv(output_path, mode='a', header=False, index=False)
        user_count += 1

print(f"\nTask Complete! Results saved to: {output_path}")

Starting Ultimate Dual-Layer Markov Model for 20000 users...
Progress: 500 / 20000...
Progress: 1000 / 20000...
Progress: 1500 / 20000...
Progress: 2000 / 20000...
Progress: 2500 / 20000...
Progress: 3000 / 20000...
Progress: 3500 / 20000...
Progress: 4000 / 20000...
Progress: 4500 / 20000...
Progress: 5000 / 20000...
Progress: 5500 / 20000...
Progress: 6000 / 20000...
Progress: 6500 / 20000...
Progress: 7000 / 20000...
Progress: 7500 / 20000...
Progress: 8000 / 20000...
Progress: 8500 / 20000...
Progress: 9000 / 20000...
Progress: 9500 / 20000...
Progress: 10000 / 20000...
Progress: 10500 / 20000...
Progress: 11000 / 20000...
Progress: 11500 / 20000...
Progress: 12000 / 20000...
Progress: 12500 / 20000...
Progress: 13000 / 20000...
Progress: 13500 / 20000...
Progress: 14000 / 20000...
Progress: 14500 / 20000...
Progress: 15000 / 20000...
Progress: 15500 / 20000...
Progress: 16000 / 20000...
Progress: 16500 / 20000...
Progress: 17000 / 20000...
Progress: 17500 / 20000...
Progress: 1800

In [2]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# --- Configuration & File Paths ---
pred_path = 'Math(markov).csv'
orig_path = '/home/z5562390/work/dataset2023/dataset1.csv.gz'
merged_output_path = 'predictions_vs_truth_markov.csv'

# 1. Load predictions and identify target users
print("1. Loading predictions into memory...")
pred_df = pd.read_csv(pred_path)
target_uids = pred_df['uid'].unique()
print(f"-> Successfully loaded predictions for {len(target_uids)} users.")

# 2. Extract Ground Truth (Days 60-74) via chunking to manage memory
print("\n2. Extracting Ground Truth (Days 60-74) from massive dataset...")
gt_chunks = []
dtypes = {'uid': np.int32, 'd': np.int16, 't': np.int8, 'x': np.int16, 'y': np.int16}

for chunk in pd.read_csv(orig_path, usecols=['uid', 'd', 't', 'x', 'y'], dtype=dtypes, chunksize=1000000):
    # Filter by target dates and target users
    chunk = chunk[(chunk['d'] >= 60) & (chunk['d'] <= 74)]
    chunk = chunk[chunk['uid'].isin(target_uids)]
    
    # Keep the final settled location per time bin
    chunk = chunk.drop_duplicates(subset=['uid', 'd', 't'], keep='last')
    
    if not chunk.empty:
        gt_chunks.append(chunk)

gt_df = pd.concat(gt_chunks, ignore_index=True)
print(f"-> Extracted {len(gt_df)} real trajectory records for evaluation.")

# 3. Merge predictions with ground truth and calculate accuracy
print("\n3. Comparing Predictions vs. Ground Truth...")
eval_df = pd.merge(gt_df, pred_df, on=['uid', 'd', 't'], suffixes=('_true', '_pred'))

# Strict evaluation: Both X and Y must match exactly
eval_df['is_correct'] = (eval_df['x_true'] == eval_df['x_pred']) & (eval_df['y_true'] == eval_df['y_pred'])

accuracy = eval_df['is_correct'].mean() * 100
total_evaluated = len(eval_df)
correct_count = eval_df['is_correct'].sum()

print("==================================================")
print("📊 EVALUATION RESULTS")
print("==================================================")
print(f"Total Evaluated Records: {total_evaluated}")
print(f"Correct Predictions:     {correct_count}")
print(f"Overall Accuracy:        {accuracy:.2f} %")
print("==================================================")

# 4. Export comparison results for downstream error analysis
print(f"\n4. Saving merged results to {merged_output_path}...")
output_cols = ['uid', 'd', 't', 'x_true', 'y_true', 'x_pred', 'y_pred']
final_export_df = eval_df[output_cols]
final_export_df.to_csv(merged_output_path, index=False)

print("-> Save complete! You can now analyze the errors in the new CSV file.")

1. Loading predictions into memory...
-> Successfully loaded predictions for 20000 users.

2. Extracting Ground Truth (Days 60-74) from massive dataset...
-> Extracted 4861176 real trajectory records for evaluation.

3. Comparing Predictions vs. Ground Truth...
📊 EVALUATION RESULTS
Total Evaluated Records: 4861176
Correct Predictions:     951434
Overall Accuracy:        19.57 %

4. Saving merged results to predictions_vs_truth_markov.csv...
-> Save complete! You can now analyze the errors in the new CSV file.
